# EE P 596 – Assignment 3
Run the cells from top to bottom with **Shift + Enter** (or **Run → Run All Cells**).
Keep this notebook in the same folder as `original_image.png` and `cat_eye.jpg`.

## Setup (run once)
Installs PyTorch only if it's missing. After a fresh install, use **Kernel → Restart Kernel**.

In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    %pip install torch torchvision

## The Assignment 3 class (same code as Assignment3.py)

In [ ]:
import numpy as np
import torch
import torchvision
import cv2 as cv
import matplotlib.pyplot as plt


class Assignment3:
    def __init__(self) -> None:
        pass

    def torch_image_conversion(self, torch_img):
        # Q1a: OpenCV image (BGR, uint8, H x W x 3) -> RGB float32 torch tensor (H x W x 3)
        rgb = cv.cvtColor(torch_img, cv.COLOR_BGR2RGB)        # swap BGR -> RGB
        torch_img = torch.from_numpy(rgb).to(torch.float32)   # numpy -> torch, as float32
        return torch_img

    def brighten(self, torch_img):
        # Q1b: add 100.0 to every value (float32, so nothing wraps around)
        bright_img = torch_img + 100.0
        return bright_img

    def saturation_arithmetic(self, img):
        # Q1c: RGB, add 100 in int16, clamp to 0..255, then back to uint8
        rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        wide = torch.from_numpy(rgb).to(torch.int16)          # wider type: can hold 355
        wide = wide + 100
        saturated_img = torch.clamp(wide, 0, 255).to(torch.uint8)
        return saturated_img

    def add_noise(self, torch_img):
        # Q2: Gaussian noise, mean 0, sigma 100 gray levels; return float32 in 0..1
        noise = torch.randn(torch_img.shape) * 100.0           # randn = mean 0, std 1
        noisy = torch_img.to(torch.float32) + noise
        noisy_img = torch.clamp(noisy / 255.0, 0.0, 1.0).to(torch.float32)
        return noisy_img

    def normalization_image(self, img):
        # Q3a: subtract the image's own per-channel mean, divide by its std (float64, RGB)
        # No clamping -> result has zero mean and unit std per channel
        rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        x = torch.from_numpy(rgb).to(torch.float64)
        mean = x.mean(dim=(0, 1))                              # 3 values: R, G, B
        std = x.std(dim=(0, 1))                                # 3 values: R, G, B
        image_norm = (x - mean) / std                          # broadcasting over H x W
        return image_norm

    def Imagenet_norm(self, img):
        # Q3b: scale to 0..1, normalize with ImageNet mean/std (RGB, float64), clamp to [0, 1]
        rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        x = torch.from_numpy(rgb).to(torch.float64) / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float64)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float64)
        ImageNet_norm = torch.clamp((x - mean) / std, 0.0, 1.0)
        return ImageNet_norm

    def dimension_rearrange(self, img):
        # Q4: H x W x C  ->  N x C x H x W  (N = 1), float32, RGB
        rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        x = torch.from_numpy(rgb).to(torch.float32)            # (H, W, C)
        rearrange = x.permute(2, 0, 1).unsqueeze(0)            # (C, H, W) -> (1, C, H, W)
        return rearrange

    def stride(self, img):
        # Q5: 3x3 Scharr_x filter, zero padding 1, stride 2, on a grayscale image
        scharr_x = torch.tensor([[-3.0, 0.0, 3.0],
                                 [-10.0, 0.0, 10.0],
                                 [-3.0, 0.0, 3.0]], dtype=torch.float32)
        scharr_x = torch.flip(scharr_x, dims=[0, 1])           # true convolution flips the kernel
        x = torch.from_numpy(img).to(torch.float32)            # (H, W)
        x = x.unsqueeze(0).unsqueeze(0)                        # (1, 1, H, W) for conv2d
        k = scharr_x.unsqueeze(0).unsqueeze(0)                 # (1, 1, 3, 3)
        out = torch.nn.functional.conv2d(x, k, stride=2, padding=1)
        stride_imgage = out.squeeze(0).squeeze(0).type(torch.FloatTensor)   # back to 2-D
        return stride_imgage

## Load the images

In [ ]:
img = cv.imread("original_image.png")
img_cat = cv.imread("cat_eye.jpg", cv.IMREAD_GRAYSCALE)
assert img is not None, "original_image.png not found - put it in the same folder as this notebook"
assert img_cat is not None, "cat_eye.jpg not found - put it in the same folder as this notebook"
assign = Assignment3()
print("Images loaded:", img.shape, img_cat.shape)

## Q1a – Convert to a PyTorch tensor

In [ ]:
torch_img = assign.torch_image_conversion(img)
print("Original OpenCV Image shape:", img.shape)
print("Original OpenCV Image dtype:", img.dtype)
print("Torch Image shape:", torch_img.shape)
print("Torch Image dtype:", torch_img.dtype)
print("Torch Image Pixel dtype:", torch_img[0, 0, 0].dtype)

plt.imshow(torch_img.numpy() / 255)
plt.title("Q1a: Torch image (RGB)")
plt.axis("off")
plt.show()

## Q1b – Brighten (+100)

In [ ]:
bright_img = assign.brighten(torch_img)
print("Torch Image shape:", bright_img.shape)
print("Torch Image dtype:", bright_img.dtype)
print("Torch Image Pixel dtype:", bright_img[0, 0, 0].dtype)
print("Max value:", bright_img.max().item())

plt.imshow((bright_img / 255).clamp(0, 1).numpy())   # clamped for display only
plt.title("Q1b: Brightened (+100)")
plt.axis("off")
plt.show()

## Q1c – Saturation arithmetic

In [ ]:
saturated_img = assign.saturation_arithmetic(img)
print("dtype:", saturated_img.dtype)
print("min:", saturated_img.min().item(), " max:", saturated_img.max().item())

plt.imshow(saturated_img.numpy())
plt.title("Q1c: Saturation arithmetic (int16 + clamp)")
plt.axis("off")
plt.show()

## Q2 – Add Gaussian noise (mean 0, σ = 100)

In [ ]:
noisy_img = assign.add_noise(torch_img)
print("dtype:", noisy_img.dtype)
print("min:", noisy_img.min().item(), " max:", noisy_img.max().item())

plt.imshow(noisy_img.numpy())
plt.title("Q2: Gaussian noise")
plt.axis("off")
plt.show()

## Q3a – Normalize with the image's own mean and std

In [ ]:
image_norm = assign.normalization_image(img)
print("dtype:", image_norm.dtype)
print("Mean per channel (should be ~0):", image_norm.mean(dim=(0, 1)))
print("Std per channel  (should be ~1):", image_norm.std(dim=(0, 1)))

show = (image_norm - image_norm.min()) / (image_norm.max() - image_norm.min())   # stretched for display only
plt.imshow(show.numpy())
plt.title("Q3a: Own-image normalization")
plt.axis("off")
plt.show()

## Q3b – ImageNet normalization

In [ ]:
ImageNet_norm = assign.Imagenet_norm(img)
print("dtype:", ImageNet_norm.dtype)
print("min:", ImageNet_norm.min().item(), " max:", ImageNet_norm.max().item())

plt.imshow(ImageNet_norm.numpy())
plt.title("Q3b: ImageNet normalization")
plt.axis("off")
plt.show()

## Q4 – Rearrange dimensions (H×W×C → N×C×H×W)
No picture – only the shape changes.

In [ ]:
rearrange = assign.dimension_rearrange(img)
print("Before (H, W, C):   ", torch_img.shape)
print("After  (N, C, H, W):", rearrange.shape)
print("dtype:", rearrange.dtype)

## Q5 – Stride convolution with Scharr_x (stride 2, zero padding)

In [ ]:
stride_imgage = assign.stride(img_cat)
print("Cat image shape:", img_cat.shape)
print("Output shape:   ", stride_imgage.shape)
print("Output type:    ", stride_imgage.type())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_cat, cmap="gray")
plt.title("Original cat eye")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(stride_imgage.abs().numpy(), cmap="gray")
plt.title("Q5: Scharr_x, stride 2")
plt.axis("off")
plt.show()

## All outputs together (handy for the report)

In [ ]:
plt.figure(figsize=(18, 8))
pics = [
    ("Q1a Tensor",        torch_img / 255),
    ("Q1b Brighten",      (bright_img / 255).clamp(0, 1)),
    ("Q1c Saturation",    saturated_img),
    ("Q2 Noise",          noisy_img),
    ("Q3a Own norm",      (image_norm - image_norm.min()) / (image_norm.max() - image_norm.min())),
    ("Q3b ImageNet norm", ImageNet_norm),
]
for i, (title, p) in enumerate(pics, 1):
    plt.subplot(2, 4, i)
    plt.imshow(p.numpy())
    plt.title(title)
    plt.axis("off")
plt.subplot(2, 4, 7)
plt.imshow(stride_imgage.abs().numpy(), cmap="gray")
plt.title("Q5 Stride")
plt.axis("off")
plt.tight_layout()
plt.show()